# Day 2 Session 3: Grid Search with Real Flu Data

**Time:** 1:00-2:15  
**Theme:** Use a simple SIR model to fit real Arizona influenza hospitalization data, then forecast the future.

In this session, we will reuse the same flu-forecast theme that will appear in the MCMC notebook on Day 3:

1. Load the real Arizona influenza hospitalization data.
2. Use the first **120 days** as training data.
3. Search over many possible values of $\beta$ and $\gamma$.
4. Choose the pair with the smallest training loss.
5. Substitute the best parameters back into the SIR model.
6. Forecast the remaining days and compare with real observations.

This notebook is meant to be an intuitive bridge:

> **Grid search** gives us one best parameter pair.  
> **MCMC**, which we will see later, explores many reasonable parameter values.


## Suggested pacing

| Time | Notebook section | Activity | Priority |
|---|---|---|---|
| 1:00-1:05 | Motivation | Why do we need parameter search? | Core |
| 1:05-1:15 | Load and visualize real data | Plot Arizona influenza hospitalizations and describe the curve | Core |
| 1:15-1:25 | Train/forecast split | Fit on the first 120 days and hold out the forecast period | Core |
| 1:25-1:35 | Review SIR model + one parameter pair | Connect $\beta$, $\gamma$, and $ho I(t)$ to hospitalization predictions | Core |
| 1:35-1:50 | Loss function and grid search | Define model-data mismatch and search over $\beta$, $\gamma$ | Core |
| 1:50-2:00 | Loss surface and best parameters | Visualize the best region and interpret the parameter pair | Core |
| 2:00-2:10 | Substitute parameters and forecast | Compare fitted/forecast curves with the observed data | Core |
| 2:10-2:15 | Exercise discussion and Day 3 connection | Change one grid-search choice or discuss MCMC as the next step | Core/Stretch |

> **Pacing note:** This session is the strongest bridge from modeling to inference. Keep the stretch exercise short so the group is ready for optimization.


## 1. Import packages

We only need common Python packages: `numpy`, `pandas`, and `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Make plots a little easier to read.
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

## 2. Load the real influenza hospitalization data

The file `AZ_DATA.csv` contains daily Arizona confirmed influenza hospitalizations from **October 1, 2023** to **April 27, 2024**.

The notebook first looks for a local file. If the local file is not available, it tries to load the data from the workshop GitHub repository.

In [ ]:
github_url = "https://raw.githubusercontent.com/AZDSC-NAU/AZDSC_NAU_SITE/master/project_1/materials/AZ_DATA.csv"
possible_paths = [
    Path("AZ_DATA.csv"),
    Path("../AZ_DATA.csv"),
    Path("/mnt/data/AZ_DATA.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is not None:
    data = pd.read_csv(data_path)
    print(f"Loaded data from local file: {data_path}")
else:
    data = pd.read_csv(github_url)
    print("Loaded data from GitHub.")

# Standardize column names for easier coding.
data["date"] = pd.to_datetime(data["date"])
data["hospitalized"] = data["total_patients_hospitalized_confirmed_influenza"].astype(float)
data["rolling_7"] = data["hospitalized"].rolling(window=7, center=True).mean()

data.head()

## 3. Visualize the data before modeling

Before fitting any model, we should first look at the data.

Discussion questions:

1. When does the hospitalization count increase most quickly?
2. When is the peak?
3. Does the decrease after the peak look smooth?
4. Why might real data differ from a simple mathematical model?

In [ ]:
peak_index = data["hospitalized"].idxmax()
peak_date = data.loc[peak_index, "date"]
peak_value = data.loc[peak_index, "hospitalized"]

plt.figure(figsize=(10, 4))
plt.plot(data["date"], data["hospitalized"], marker="o", markersize=3, linewidth=1, label="Daily hospitalizations")
plt.plot(data["date"], data["rolling_7"], linewidth=2, label="7-day rolling average")
plt.scatter([peak_date], [peak_value], s=80, label=f"Peak: {int(peak_value)}")
plt.title("Arizona Influenza Hospitalizations")
plt.xlabel("Date")
plt.ylabel("Hospitalizations")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Peak hospitalization count: {int(peak_value)} on {peak_date.date()}")
print(f"Number of days in the dataset: {len(data)}")

## 4. Split the data into training and forecast periods

A simple but important forecasting idea is:

> Fit the model using the past, then test the forecast on the future.

We will use the first **120 days** as training data. The remaining days are held out until after we estimate the model parameters.

In [ ]:
y = data["hospitalized"].to_numpy()
dates = data["date"].to_numpy()
n_days = len(y)

train_end = 120
train_y = y[:train_end]
forecast_y = y[train_end:]

print(f"Training days: 0 to {train_end - 1}")
print(f"Forecast days: {train_end} to {n_days - 1}")
print(f"Training period ends on: {pd.to_datetime(dates[train_end-1]).date()}")
print(f"Forecast period begins on: {pd.to_datetime(dates[train_end]).date()}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(data["date"], y, marker="o", markersize=3, linewidth=1, label="Observed hospitalizations")
plt.axvline(data["date"].iloc[train_end], linestyle="--", label="Forecast begins")
plt.title("Training Period and Forecast Period")
plt.xlabel("Date")
plt.ylabel("Hospitalizations")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Review the SIR model

The SIR model is

$$
\begin{aligned}
\frac{dS}{dt} &= -\frac{\beta S I}{N}, \\
\frac{dI}{dt} &= \frac{\beta S I}{N} - \gamma I, \\
\frac{dR}{dt} &= \gamma I.
\end{aligned}
$$

Here:

- $S(t)$: susceptible population,
- $I(t)$: infected population,
- $R(t)$: recovered population,
- $\beta$: transmission rate,
- $\gamma$: recovery rate.

The observed data are **hospitalizations**, not total infections. To keep the workshop example simple, we use

$$
\text{predicted hospitalizations}(t) = \rho I(t),
$$

where $\rho$ is a scaling factor. For each proposed $(\beta,\gamma)$, we choose the best $\rho$ automatically.

In [ ]:
def sir_rhs(state, beta, gamma, N):
    """Right-hand side of the SIR model."""
    S, I, R = state
    dS = -beta * S * I / N
    dI = beta * S * I / N - gamma * I
    dR = gamma * I
    return np.array([dS, dI, dR])


def simulate_sir(beta, gamma, n_days, N=100_000, I0=10, dt=0.25):
    """Simulate the SIR model and return daily infected values I(t)."""
    S0 = N - I0
    R0 = 0.0
    state = np.array([S0, I0, R0], dtype=float)

    steps_per_day = int(round(1 / dt))
    infected_daily = np.zeros(n_days)

    for day in range(n_days):
        infected_daily[day] = state[1]
        for _ in range(steps_per_day):
            state = state + dt * sir_rhs(state, beta, gamma, N)
            state = np.maximum(state, 0)  # avoid tiny negative values from numerical error

    return infected_daily


def best_scale_factor(model_infected, observed):
    """Choose rho to make rho * I(t) close to the observed hospitalizations."""
    denominator = np.sum(model_infected ** 2)
    if denominator == 0:
        return 0.0
    rho = np.sum(observed * model_infected) / denominator
    return max(rho, 0.0)


def predict_hospitalizations(beta, gamma, n_days, observed_for_scale=None):
    """Run SIR and convert infected values to predicted hospitalizations."""
    infected = simulate_sir(beta, gamma, n_days)

    if observed_for_scale is None:
        rho = 1.0
    else:
        rho = best_scale_factor(infected[:len(observed_for_scale)], observed_for_scale)

    predicted = rho * infected
    return predicted, rho

## 6. Try one parameter pair

Before doing grid search, let's try one pair of parameters and see what the model predicts.

In [ ]:
example_beta = 0.20
example_gamma = 0.10

example_pred, example_rho = predict_hospitalizations(
    example_beta,
    example_gamma,
    n_days,
    observed_for_scale=train_y
)

plt.figure(figsize=(10, 4))
plt.plot(data["date"], y, marker="o", markersize=3, linewidth=1, label="Observed data")
plt.plot(data["date"], example_pred, linewidth=2, label="Example model prediction")
plt.axvline(data["date"].iloc[train_end], linestyle="--", label="Forecast begins")
plt.title(f"Example Parameters: beta={example_beta}, gamma={example_gamma}, rho={example_rho:.4f}")
plt.xlabel("Date")
plt.ylabel("Hospitalizations")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Define the loss function

We use the training data only:

$$
\text{Loss}(\beta,\gamma)
=
\frac{1}{T}\sum_{t=1}^{T}
\left(y_t - \widehat{y}_t(\beta,\gamma)\right)^2.
$$

This is the mean squared error between the observed hospitalizations and the model-predicted hospitalizations.

A smaller loss means the model fits the training data better.

In [ ]:
def loss(beta, gamma, observed):
    """Mean squared error on the observed training data."""
    predicted, rho = predict_hospitalizations(
        beta,
        gamma,
        len(observed),
        observed_for_scale=observed
    )
    mse = np.mean((observed - predicted) ** 2)
    return mse, rho

example_loss, example_rho = loss(example_beta, example_gamma, train_y)
print(f"Example training loss: {example_loss:.2f}")
print(f"Example scale factor rho: {example_rho:.4f}")

## 8. Grid search over $\beta$ and $\gamma$

Grid search means we create a table of possible parameter values and try every combination.

For this workshop example, we search over:

$$
\beta \in [0.01, 0.25],
\qquad
\gamma \in [0.005, 0.20].
$$

For each pair:

1. Run the SIR model.
2. Choose the best scale factor $\rho$.
3. Compute the training loss.
4. Keep track of the best pair.

In [ ]:
beta_values = np.linspace(0.01, 0.25, 49)
gamma_values = np.linspace(0.005, 0.20, 40)

loss_grid = np.zeros((len(gamma_values), len(beta_values)))
rho_grid = np.zeros_like(loss_grid)

for i, gamma in enumerate(gamma_values):
    for j, beta in enumerate(beta_values):
        loss_grid[i, j], rho_grid[i, j] = loss(beta, gamma, train_y)

best_index = np.unravel_index(np.argmin(loss_grid), loss_grid.shape)
best_gamma = gamma_values[best_index[0]]
best_beta = beta_values[best_index[1]]
best_loss = loss_grid[best_index]
best_rho = rho_grid[best_index]

print("Best grid-search result")
print(f"beta  = {best_beta:.4f}")
print(f"gamma = {best_gamma:.4f}")
print(f"rho   = {best_rho:.4f}")
print(f"training MSE = {best_loss:.2f}")
print(f"training RMSE = {np.sqrt(best_loss):.2f} hospitalizations")

## 9. Visualize the loss surface

A loss surface shows which parameter pairs fit well and which fit poorly.

The best pair is marked on the plot.

In [ ]:
from matplotlib.colors import LogNorm

B, G = np.meshgrid(beta_values, gamma_values)

plt.figure(figsize=(8, 6))
plt.contourf(B, G, loss_grid, levels=30, norm=LogNorm())
plt.colorbar(label="Training loss, log scale")
plt.scatter([best_beta], [best_gamma], s=80, edgecolors="white", label="Best pair")
plt.title("Grid Search Loss Surface")
plt.xlabel(r"$\beta$ transmission rate")
plt.ylabel(r"$\gamma$ recovery rate")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Substitute the best parameters back into the model

Now we use the best $\beta$, $\gamma$, and $\rho$ to generate predictions for the whole time period.

The model is only fit to the first 120 days. The remaining days are used to evaluate the forecast.

In [ ]:
best_pred, best_rho_check = predict_hospitalizations(
    best_beta,
    best_gamma,
    n_days,
    observed_for_scale=train_y
)

train_rmse = np.sqrt(np.mean((train_y - best_pred[:train_end]) ** 2))
forecast_rmse = np.sqrt(np.mean((forecast_y - best_pred[train_end:]) ** 2))

print(f"Training RMSE: {train_rmse:.2f} hospitalizations")
print(f"Forecast RMSE: {forecast_rmse:.2f} hospitalizations")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(data["date"], y, marker="o", markersize=3, linewidth=1, label="Observed data")
plt.plot(data["date"], best_pred, linewidth=2, label="Grid-search forecast")
plt.axvline(data["date"].iloc[train_end], linestyle="--", label="Forecast begins")
plt.title("Grid Search Fit and Forecast")
plt.xlabel("Date")
plt.ylabel("Hospitalizations")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

### Discussion pause

1. Does the model match the training period well?
2. Does the forecast capture the downward trend?
3. Where does the model miss the real data?
4. What real-world factors might a simple SIR model be missing?

## 11. Compare observed and predicted values in the forecast period

This plot focuses only on the held-out forecast period.

In [ ]:
forecast_dates = data["date"].iloc[train_end:]
forecast_pred = best_pred[train_end:]

plt.figure(figsize=(10, 4))
plt.plot(forecast_dates, forecast_y, marker="o", markersize=3, linewidth=1, label="Observed future data")
plt.plot(forecast_dates, forecast_pred, linewidth=2, label="Forecast from grid-search parameters")
plt.title("Forecast Period Only")
plt.xlabel("Date")
plt.ylabel("Hospitalizations")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 12. Exercise: Explore grid-search choices

### Core exercise

Change one number at a time and rerun the notebook cells.

1. Change `train_end` from `120` to `100` or `140`.
2. Change the number of grid points in

```python
beta_values = np.linspace(0.01, 0.25, 49)
gamma_values = np.linspace(0.005, 0.20, 40)
```

For example, try fewer grid points:

```python
beta_values = np.linspace(0.01, 0.25, 25)
gamma_values = np.linspace(0.005, 0.20, 20)
```

Then answer:

- Did the best parameters change?
- Did the forecast change?
- Was the computation faster?

### Stretch exercise

Try expanding the search range:

```python
beta_values = np.linspace(0.01, 0.35, 60)
gamma_values = np.linspace(0.005, 0.25, 50)
```

Then answer:

- Does the best pair move to the edge of the plot?
- What does it mean if the best pair is on the edge of the grid?

In [ ]:
# Solution workspace: try a shorter grid and compare the result.
beta_values_ex = np.linspace(0.01, 0.25, 25)
gamma_values_ex = np.linspace(0.005, 0.20, 20)

loss_grid_ex = np.zeros((len(gamma_values_ex), len(beta_values_ex)))
for i, gamma in enumerate(gamma_values_ex):
    for j, beta in enumerate(beta_values_ex):
        loss_grid_ex[i, j], _ = loss(beta, gamma, train_y)

best_index_ex = np.unravel_index(np.argmin(loss_grid_ex), loss_grid_ex.shape)
best_gamma_ex = gamma_values_ex[best_index_ex[0]]
best_beta_ex = beta_values_ex[best_index_ex[1]]
best_loss_ex = loss_grid_ex[best_index_ex]

print("Shorter-grid result")
print(f"beta  = {best_beta_ex:.4f}")
print(f"gamma = {best_gamma_ex:.4f}")
print(f"training RMSE = {np.sqrt(best_loss_ex):.2f} hospitalizations")
print("Compare these values with the original grid-search result above.")


## 13. Connection to Day 3 MCMC

Grid search and MCMC both use the idea of scoring parameter values.

- **Grid search:** tries a fixed table of parameter values and chooses the best one.
- **MCMC:** walks randomly through the parameter space and spends more time near better parameter values.

For forecasting, grid search gives one parameter pair, while MCMC can give many plausible parameter pairs. That allows us to discuss uncertainty in the forecast.